# Week 1: Python and Environment Review
## From a raw flight log to engineering answers

**Course:** AI for Aerospace & Mechanical Engineering (PhD) &nbsp;|&nbsp; **Block 0:** Python & OOP crash course

> **Provenance:** new content. The instructor's "environment setup + ML intro" PDF was not
> available when this notebook was written; reconcile with it when it is.
> Python/NumPy/Pandas/Matplotlib explanations are rewritten from (not copied from):
> [W3Schools NumPy intro](https://www.w3schools.com/python/numpy/numpy_intro.asp),
> [W3Schools Pandas Read CSV](https://www.w3schools.com/python/pandas/pandas_csv.asp),
> [W3Schools Pandas Analyzing DataFrames](https://www.w3schools.com/python/pandas/pandas_analyzing.asp),
> [W3Schools Matplotlib Pyplot](https://www.w3schools.com/python/matplotlib_pyplot.asp),
> [GeeksforGeeks DataFrame.describe()](https://www.geeksforgeeks.org/python/python-pandas-dataframe-describe-method/),
> [GeeksforGeeks Anaconda virtual environments](https://www.geeksforgeeks.org/python/set-up-virtual-environment-for-python-using-anaconda/),
> [GeeksforGeeks Python virtual environments](https://www.geeksforgeeks.org/python/python-virtual-environment/),
> [GeeksforGeeks Getting started with Jupyter](https://www.geeksforgeeks.org/python/getting-started-with-jupyter-notebook-python/).
> Load-factor relations adapted from Anderson, *Introduction to Flight*, Ch. 6.

**How to use this notebook.** Cells marked `# TODO` are yours. Each exercise ends with a
`check(...)` call that prints `[ ]` until you fill it in and `[x]` when your answer is right.
The notebook runs top to bottom even before you solve anything.

## 0. The problem

A flight-test engineer hands you `week01_uav_flight_log.csv`: five minutes of a small
fixed-wing UAV test flight. The IMU (3-axis accelerometer and gyroscope) is logged at 50 Hz
together with barometric altitude, pitot airspeed and outside air temperature.

Before any machine learning can happen, the chief engineer wants four answers:

| # | Question | Why it matters |
|---|----------|----------------|
| Q1 | What was the sampling rate, and did the logger lose data? | Every later algorithm assumes a known, uniform time step. |
| Q2 | What was the peak load factor? Did the airframe exceed its **2.0 g** limit? | A "yes" grounds the aircraft for inspection. |
| Q3 | How noisy are the gyros while the aircraft is parked? | The navigation team needs the noise level to tune their Kalman filter. |
| Q4 | What was the peak climb rate? | Performance check against the design value (~15 m/s). |

**Today's tools, in the order we need them:** a reproducible environment (conda/venv, Jupyter,
Git) → NumPy for vectorised maths → Pandas for tables → Matplotlib for plots.

## 1. The environment

Every result in this course must be reproducible on another machine. That is the job of an
*isolated environment*: a private Python installation with pinned package versions.

```bash
# Option A: conda (recommended on Windows; handles PyTorch/CUDA later)
conda env create -f environment.yml     # or: conda create -n aero-ai python=3.11
conda activate aero-ai
conda env list                          # which environments exist?
conda list numpy                        # which numpy version is installed?

# Option B: built-in venv + pip
python -m venv .venv
source .venv/bin/activate               # Windows: .venv\Scripts\activate
pip install -r requirements.txt
deactivate
```

```bash
# Git: the minimum you need this week
git clone https://github.com/CodeNomad84/AI_ML_Classrom.git
git switch -c week01-<your-name>        # your own branch
git status                              # what changed?
git add notebooks/week01/week01_python_env.ipynb
git commit -m "Week 1: answer Q1-Q4"
git push -u origin week01-<your-name>
```

Run the next cell. It finds the repository root, makes the course package importable, and
defines the constants we use (no magic numbers further down).

In [ ]:
from __future__ import annotations

import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Callable, Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_PATH: Final[Path] = REPO_ROOT / "data" / "sample" / "week01_uav_flight_log.csv"
G0: Final[float] = 9.80665          # standard gravity [m/s^2]
LOAD_LIMIT_G: Final[float] = 2.0    # structural limit of the airframe [g]
GROUND_END_S: Final[float] = 25.0   # aircraft is parked before this time [s]


def check(name: str, value: object, expected: float, tol: float) -> None:
    """Print a pass mark if ``value`` is within ``tol`` of ``expected``.

    A value of None means the exercise has not been attempted yet.
    """
    if value is None:
        print(f"[ ] {name}: not done yet")
        return
    assert abs(float(value) - expected) <= tol, f"{name}: got {value}, expected ~{expected}"
    print(f"[x] {name} = {float(value):.4g}")


print("Repository root found:", REPO_ROOT.name)
print("Data file exists:", DATA_PATH.exists())

In [ ]:
# Compare installed versions with the course pins (major.minor).
COURSE_PINS: Final[dict[str, str]] = {
    "numpy": "1.26", "pandas": "2.1", "matplotlib": "3.8",
}

print(f"python      {sys.version.split()[0]:<10} (course pin 3.11)")
for package, pin in COURSE_PINS.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = "MISSING"
    flag = "ok" if installed.startswith(pin) else "differs from pin"
    print(f"{package:<11} {installed:<10} (course pin {pin}) {flag}")

A "differs from pin" line is a warning, not an error: this notebook avoids version-specific
APIs. For graded work, use the course environment so your results match the instructor's.

## 2. Why NumPy? A one-minute experiment

A Python list is a list of *objects*; a NumPy array is one contiguous block of numbers of a
single type. Operations on the whole array run in compiled code ("vectorisation").
We compute the magnitude of 200 000 acceleration vectors both ways.

In [ ]:
N_VECTORS: Final[int] = 200_000
rng = np.random.default_rng(seed=0)
accel = rng.normal(0.0, 1.0, size=(N_VECTORS, 3))   # shape (N, 3)
accel_as_lists = accel.tolist()

TIMING_REPEATS: Final[int] = 5


def best_time_s(func: Callable[[], object], repeats: int = TIMING_REPEATS) -> float:
    """Return the fastest of repeats runs (the first run includes warm-up costs)."""
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        func()
        timings.append(time.perf_counter() - start)
    return min(timings)


norms_loop = [(ax**2 + ay**2 + az**2) ** 0.5 for ax, ay, az in accel_as_lists]
norms_vec = np.sqrt(np.sum(accel**2, axis=1))        # whole-array operations
loop_s = best_time_s(lambda: [(ax**2 + ay**2 + az**2) ** 0.5 for ax, ay, az in accel_as_lists])
vec_s = best_time_s(lambda: np.sqrt(np.sum(accel**2, axis=1)))

assert np.allclose(norms_loop, norms_vec)
print(f"Python loop: {loop_s * 1e3:7.1f} ms")
print(f"NumPy      : {vec_s * 1e3:7.1f} ms  (~{loop_s / vec_s:.0f}x faster)")

**Shapes and axes.** `accel` has shape `(N, 3)`: axis 0 runs over samples, axis 1 over the
x, y, z components. `np.linalg.norm(accel, axis=1)` collapses axis 1, so the result has shape
`(N,)`: one magnitude per sample. Getting `axis` right is the most common NumPy bug.

## 3. Worked Example 1 (by hand first): noise statistics and `ddof`

Five gyroscope z-axis samples taken while the aircraft is parked, in deg/s:

$$x = [0.2,\ -0.4,\ 0.1,\ 0.5,\ -0.4]$$

**By hand.**
Mean: $\bar x = (0.2 - 0.4 + 0.1 + 0.5 - 0.4)/5 = 0.0/5 = 0.0$.
Squared deviations: $0.04,\ 0.16,\ 0.01,\ 0.25,\ 0.16$; their sum is $0.62$.

* Population variance (divide by $N$): $0.62/5 = 0.124 \Rightarrow \sigma = 0.3521$
* Sample variance (divide by $N-1$, unbiased): $0.62/4 = 0.155 \Rightarrow s = 0.3937$

The two differ by 12 % with five samples. **NumPy's `np.std` divides by $N$ by default
(`ddof=0`); Pandas' `.std()` divides by $N-1$ (`ddof=1`).** Same data, two libraries, two
answers, unless you set `ddof` explicitly.

In [ ]:
gyro_z_rest = np.array([0.2, -0.4, 0.1, 0.5, -0.4])

print("mean            :", gyro_z_rest.mean())
print("np.std (ddof=0) :", round(np.std(gyro_z_rest), 4))
print("np.std (ddof=1) :", round(np.std(gyro_z_rest, ddof=1), 4))
print("pd .std()       :", round(pd.Series(gyro_z_rest).std(), 4))

assert np.isclose(np.std(gyro_z_rest), 0.3521, atol=1e-4)
assert np.isclose(pd.Series(gyro_z_rest).std(), 0.3937, atol=1e-4)

**Exercise 3.1.** Five accelerometer z samples at rest: `[9.70, 9.90, 9.80, 9.85, 9.75]` m/s².
Compute the mean and the **sample** standard deviation (`ddof=1`) with NumPy.
Do it on paper first: you should get a mean of 9.80 and a standard deviation of about 0.079.

In [ ]:
accel_z_rest = np.array([9.70, 9.90, 9.80, 9.85, 9.75])
accel_z_mean = None   # TODO: mean with NumPy
accel_z_std = None    # TODO: sample standard deviation (ddof=1)

check("Exercise 3.1 mean", accel_z_mean, 9.80, 1e-6)
check("Exercise 3.1 std", accel_z_std, 0.0791, 1e-4)

## 4. Worked Example 2 (by hand first): load factor from the accelerometer

An accelerometer measures **specific force** $\mathbf f$: everything except gravity. Parked on
the ground it reads $+g$ upward. The **load factor** is

$$n = \frac{\lVert \mathbf f \rVert}{g_0}, \qquad g_0 = 9.80665\ \text{m/s}^2 .$$

In a level coordinated turn with bank angle $\phi$, lift balances weight vertically
($L\cos\phi = W$), so $n = L/W = 1/\cos\phi$: a 30° bank gives $n = 1.155$, a 60° bank gives
$n = 2$.

**By hand.** One sample during a pull-up reads $\mathbf f = (1.2,\ -0.4,\ 19.6)$ m/s².

$$\lVert \mathbf f \rVert^2 = 1.2^2 + 0.4^2 + 19.6^2 = 1.44 + 0.16 + 384.16 = 385.76$$
$$\lVert \mathbf f \rVert = 19.641\ \text{m/s}^2, \qquad n = 19.641 / 9.80665 = 2.003$$

That single sample is above the 2.0 g limit. Whether the *airframe* really exceeded it is a
different question; we return to it in Q2.

In [ ]:
f = np.array([1.2, -0.4, 19.6])
n_single = np.linalg.norm(f) / G0
print(f"n = {n_single:.4f}")
assert np.isclose(n_single, 2.0028, atol=1e-4)

**Exercise 4.1.** Compute the load factor of all three samples at once, **without a loop**.
Rows are samples, columns are x, y, z. Hint: which `axis`?
Expected: `[1.000, 2.003, 1.150]`.

In [ ]:
samples = np.array([
    [0.3, 0.1, 9.8],
    [1.2, -0.4, 19.6],
    [0.0, 2.5, 11.0],
])
n_samples = None   # TODO: one NumPy expression, result shape (3,)

check("Exercise 4.1 n[0]", None if n_samples is None else n_samples[0], 1.0, 1e-3)
check("Exercise 4.1 n[2]", None if n_samples is None else n_samples[2], 1.1503, 1e-3)

## 5. Load the flight log with Pandas

`pd.read_csv` turns the file into a `DataFrame`: a table whose columns are NumPy arrays
with names. Three calls give a first impression of any new dataset:
`head()` (does it look right?), `info()` (types, non-null counts) and `describe()`
(statistics per column).

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.round(3)

**Read the table like an engineer.**

* `count` is 14 940 for most columns but 14 865 for `baro_alt_m` and `airspeed_mps`:
  **75 values are missing** in each.
* `accel_z_mps2` has a median of ≈ 9.82 (1 g, as expected) but a **max of 50.33 m/s² ≈ 5.1 g**.
  Either the UAV pulled 5 g, or the sensor glitched. Keep this in mind for Q2.
* `temp_c` falls from ≈ 15 °C to ≈ 11 °C: the standard atmosphere lapse rate
  (6.5 °C per km) times 600 m gives 3.9 °C. Consistent.

## 6. Q1: sampling rate and data loss

**Worked Example 3 (by hand).** Five consecutive timestamps from the log:
`[99.94, 99.96, 99.98, 101.20, 101.22]` s.

* Differences: `0.02, 0.02, 1.22, 0.02` s.
* The **median** difference is 0.02 s → $f_s = 1/0.02 = 50$ Hz. (The mean would be
  0.32 s, ruined by one gap; the median is robust.)
* Nyquist frequency $f_s/2 = 25$ Hz: vibrations faster than that cannot be seen in this log.
* The 1.22 s step spans $1.22/0.02 = 61$ nominal steps, so **60 samples are missing**.

In [ ]:
GAP_TOLERANCE: Final[float] = 1.5   # a step > 1.5 nominal steps is a gap

dt = df["time_s"].diff()
nominal_dt = dt.median()
sample_rate_hz = 1.0 / nominal_dt

is_gap = dt > GAP_TOLERANCE * nominal_dt
gaps = pd.DataFrame({
    "gap_start_s": df["time_s"].shift(1)[is_gap],
    "gap_end_s": df["time_s"][is_gap],
    "missing_samples": (dt[is_gap] / nominal_dt).round().astype(int) - 1,
})

print(f"Sample rate: {sample_rate_hz:.1f} Hz, Nyquist: {sample_rate_hz / 2:.1f} Hz")
gaps

**Exercise 6.1.** Besides the gap, some *values* are missing (NaN). Compute the percentage
of missing values per column as a Pandas Series, sorted from most to least missing.
Hint: `isna()` gives booleans and the mean of booleans is a fraction.
Expected: about 0.50 % for `baro_alt_m` and `airspeed_mps`, 0 % elsewhere.

In [ ]:
missing_pct = None   # TODO: Series of percentages, sorted descending

check("Exercise 6.1 baro %", None if missing_pct is None else missing_pct["baro_alt_m"], 0.502, 0.01)
missing_pct

**Answer to Q1.** The logger sampled at 50 Hz (Nyquist 25 Hz). It lost one block of
60 samples (1.2 s) starting at t ≈ 100 s, and 0.5 % of barometer and airspeed values are NaN.

## 7. Q2: peak load factor and the 2.0 g limit

In [ ]:
ACCEL_COLUMNS: Final[list[str]] = ["accel_x_mps2", "accel_y_mps2", "accel_z_mps2"]

df["n_raw"] = np.linalg.norm(df[ACCEL_COLUMNS].to_numpy(), axis=1) / G0
peak = df["n_raw"].idxmax()
print(f"Raw peak: n = {df.loc[peak, 'n_raw']:.2f} g at t = {df.loc[peak, 'time_s']:.2f} s")
df.loc[peak - 3 : peak + 3, ["time_s", "accel_z_mps2", "n_raw"]]

A 5.1 g peak would ground the aircraft, but look at its neighbours: 1.06 g, 1.06 g,
**5.13 g**, 1.06 g, 1.06 g. A real 5 g manoeuvre takes seconds to build up and decay; this
lasts one sample (20 ms). It is a **sensor glitch** (a spike), not a manoeuvre.

**Worked Example 4 (by hand): mean versus median.** A five-sample window around the spike:
`[1.00, 1.01, 5.10, 0.99, 1.00]`.

* Mean $= 9.10/5 = 1.82$ g: the spike drags the average up by 0.8 g.
* Median (sort and take the middle): `[0.99, 1.00, 1.00, 1.01, 5.10]` → **1.00 g**.

A *rolling median* slides that window along the signal: isolated spikes disappear while a
pull-up lasting 200 samples passes through unchanged.

In [ ]:
MEDIAN_WINDOW: Final[int] = 5   # samples; must be odd so the window is centred

df["n_median"] = df["n_raw"].rolling(MEDIAN_WINDOW, center=True, min_periods=1).median()
peak = df["n_median"].idxmax()
peak_n = df.loc[peak, "n_median"]
print(f"Filtered peak: n = {peak_n:.2f} g at t = {df.loc[peak, 'time_s']:.2f} s")
print("Limit exceeded:", bool(peak_n > LOAD_LIMIT_G))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df["time_s"], df["n_raw"], lw=0.6, color="0.6", label="raw")
ax.plot(df["time_s"], df["n_median"], lw=1.0, color="tab:red", label="rolling median (5)")
ax.axhline(LOAD_LIMIT_G, ls="--", color="k", label="2.0 g limit")
ax.set(xlabel="Time [s]", ylabel="Load factor n [g]", title="Q2: load factor")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.show()

**Exercise 7.1.** How long (in seconds) did the filtered load factor stay **above 1.5 g**?
Count the samples and multiply by the nominal time step. Expected: about 1.66 s
(the pull-up around t ≈ 192 s).

In [ ]:
HIGH_G_THRESHOLD: Final[float] = 1.5
time_above_1p5g_s = None   # TODO: duration in seconds

check("Exercise 7.1 duration", time_above_1p5g_s, 1.66, 0.05)

**Answer to Q2.** The raw maximum of 5.1 g is a set of single-sample sensor spikes
(there are three in this log). After a 5-sample rolling median the true peak is **1.80 g**
during the pull-up at t ≈ 192 s. The 2.0 g limit was **not** exceeded.

## 8. Q3: gyro noise while parked

The aircraft is stationary for the first 25 s, so whatever the gyros report there is noise
(plus bias). A boolean mask selects those rows.

In [ ]:
GYRO_COLUMNS: Final[list[str]] = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]

on_ground = df["time_s"] < GROUND_END_S
gyro_rest = df.loc[on_ground, GYRO_COLUMNS]
noise = pd.DataFrame({"bias_dps": gyro_rest.mean(), "std_dps": gyro_rest.std(ddof=1)})
print(f"{on_ground.sum()} samples at rest")
noise.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, column in zip(axes, GYRO_COLUMNS):
    values = gyro_rest[column]
    ax.hist(values, bins=40, density=True, alpha=0.6)
    grid = np.linspace(values.min(), values.max(), 200)
    mu, sigma = values.mean(), values.std(ddof=1)
    pdf = np.exp(-0.5 * ((grid - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
    ax.plot(grid, pdf, color="k", lw=1)
    ax.set(title=f"{column}  (s = {sigma:.2f})", xlabel="deg/s")
axes[0].set_ylabel("density")
fig.suptitle("Q3: gyro noise at rest with fitted Gaussian")
fig.tight_layout()
plt.show()

**Exercise 8.1.** Use the same `on_ground` mask to compute the mean of `accel_z_mps2` at rest.
It should match standard gravity to within 0.01 m/s²; this is how accelerometers are
sanity-checked in the field.

In [ ]:
accel_z_rest_mean = None   # TODO: mean of accel_z while parked

check("Exercise 8.1 accel_z at rest", accel_z_rest_mean, G0, 0.01)

**Answer to Q3.** All three gyros show white noise with $s \approx 0.30$ deg/s and a bias
below 0.02 deg/s; the histograms are close to Gaussian. That is the number the navigation team
needs.

## 9. Q4: climb rate from barometric altitude

**Worked Example 5 (by hand): central difference.**
Three altitude samples: $h = [250.0,\ 250.3,\ 250.5]$ m at $t = [80.00,\ 80.02,\ 80.04]$ s.

$$\dot h(80.02) \approx \frac{h(80.04) - h(80.00)}{80.04 - 80.00} = \frac{0.5}{0.04} = 12.5\ \text{m/s}$$

Now add the barometer's noise ($\sigma \approx 0.8$ m) to each sample. The difference of two
noisy samples has standard deviation $\sqrt2 \cdot 0.8 = 1.13$ m; divided by 0.04 s that is
**±28 m/s** of noise on a 15 m/s signal. Differentiation amplifies noise, so we **smooth
first, then differentiate**.

In [ ]:
SMOOTHING_WINDOW_S: Final[float] = 2.0

t = df["time_s"].to_numpy()
altitude = df["baro_alt_m"].to_numpy()
valid = ~np.isnan(altitude)
raw_rate = np.gradient(np.interp(t, t[valid], altitude[valid]), t)

# Put the samples back on a uniform 50 Hz grid: NaN dropouts and the 1.2 s gap are
# filled by linear interpolation, so a centred window is symmetric everywhere.
grid = t[0] + nominal_dt * np.arange(int(round((t[-1] - t[0]) / nominal_dt)) + 1)
altitude_grid = np.interp(grid, t[valid], altitude[valid])
window = int(round(SMOOTHING_WINDOW_S / nominal_dt))
smooth_alt = pd.Series(altitude_grid).rolling(window, center=True, min_periods=1).mean()
climb_rate = pd.Series(np.interp(t, grid, np.gradient(smooth_alt.to_numpy(), nominal_dt)),
                       index=df.index)

peak = climb_rate.idxmax()
print(f"Raw-derivative noise in cruise: {raw_rate[(df['time_s'] > 130) & (df['time_s'] < 200)].std():.1f} m/s")
print(f"Peak climb rate: {climb_rate[peak]:.1f} m/s at t = {df.loc[peak, 'time_s']:.1f} s")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df["time_s"], raw_rate, lw=0.3, color="0.75", label="raw derivative")
ax.plot(df["time_s"], climb_rate, lw=1.2, color="tab:blue", label="smoothed (2 s) derivative")
ax.set(xlabel="Time [s]", ylabel="Climb rate [m/s]", ylim=(-40, 40), title="Q4: climb rate")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.show()

**Exercise 9.1.** Find the peak **descent** rate (most negative climb rate) and when it
happens. Expected: about −20 m/s at t ≈ 239 s.

In [ ]:
peak_descent_mps = None   # TODO: most negative value of climb_rate
peak_descent_time_s = None   # TODO: time at which it occurs

check("Exercise 9.1 descent rate", peak_descent_mps, -19.78, 0.1)
check("Exercise 9.1 time", peak_descent_time_s, 238.96, 0.5)

**Answer to Q4.** The peak climb rate is ≈ 16 m/s at t ≈ 84 s, close to the 15 m/s design
value (the residual 1 m/s is barometer noise that survives the 2 s window).

## 10. One dashboard for the chief engineer

`plt.subplots(rows, cols, sharex=True)` gives aligned panels with one time axis.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
panels = [
    (df["baro_alt_m"], "Altitude [m]", "tab:blue"),
    (df["airspeed_mps"], "Airspeed [m/s]", "tab:orange"),
    (df["n_median"], "Load factor [g]", "tab:red"),
    (climb_rate, "Climb rate [m/s]", "tab:green"),
]
for ax, (series, label, color) in zip(axes, panels):
    ax.plot(df["time_s"], series, lw=0.8, color=color)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
for _, row in gaps.iterrows():
    for ax in axes:
        ax.axvspan(row["gap_start_s"], row["gap_end_s"], color="k", alpha=0.15)
axes[2].axhline(LOAD_LIMIT_G, ls="--", color="k", lw=0.8)
axes[-1].set_xlabel("Time [s]")
fig.suptitle("UAV test flight: overview (grey band = logger gap)")
fig.tight_layout()
plt.show()

## 11. The same answers through a class (preview of Weeks 2 and 3)

Everything above was procedural: variables in a notebook. The course package wraps the
same logic in a reusable `FlightLog` class (`src/aerospace_ai/utils.py`). Read its source after
class; notice:

* `@classmethod from_csv(...)`: an alternative constructor.
* `@property sample_rate_hz`: computed on access, read like an attribute.
* `__slots__`: no accidental new attributes, smaller objects.
* `__repr__`: the one-line summary printed below.
* Type hints and Google-style docstrings on every method.

In [ ]:
from aerospace_ai.utils import FlightLog

log = FlightLog.from_csv(DATA_PATH)
print(log)
print("Gaps:\n", log.time_gaps())
print(f"Peak filtered load factor: {log.robust_load_factor().max():.2f} g")
print(f"Gyro-z noise at rest: {log.segment(0, GROUND_END_S).data['gyro_z_dps'].std():.3f} deg/s")
print(f"Peak climb rate: {log.climb_rate(window_s=SMOOTHING_WINDOW_S).max():.1f} m/s")

## 12. Self-check

The cell below asserts the four answers. If it passes, your environment and the data are
consistent with the instructor's.

In [ ]:
assert np.isclose(sample_rate_hz, 50.0)
assert len(gaps) == 1 and gaps["missing_samples"].iloc[0] == 60
assert np.isclose(df["n_median"].max(), 1.80, atol=0.02) and df["n_median"].max() < LOAD_LIMIT_G
assert np.allclose(noise["std_dps"], 0.30, atol=0.03)
assert 14.0 < climb_rate.max() < 18.0
assert np.isclose(log.robust_load_factor().max(), df["n_median"].max())
print("All Week 1 checks passed.")

## What we now know how to do

* Create and activate an isolated environment (conda or venv) and check package versions.
* Use Git to branch, commit and push a notebook.
* Replace Python loops with vectorised NumPy expressions and choose the right `axis`.
* Load a sensor CSV with Pandas and read `head`/`info`/`describe` like an engineer.
* Recover the sample rate robustly and locate logger gaps and NaN dropouts.
* Tell a sensor spike from a real manoeuvre and remove it with a rolling median.
* Estimate noise (`ddof`!) and a derivative (smooth first) from noisy measurements.
* Present the results in a multi-panel Matplotlib figure.

**Next week:** we turn `FlightLog`'s ideas into a `Sensor` class hierarchy (IMU, LiDAR, GPS)
with a shared `preprocess()` interface, and test it with `pytest`.